# 10 — Decision rule and threshold (Phase 10)

The model gives each candidate pair a **probability** of being the same business. We still have to **decide** which
IDs to output for each S1 entity. That decision is what the leaderboard scores.

So far: *predict every pair with probability ≥ 0.70* (one global threshold, chosen on the tuning entities) → validation F0.5 **0.934**.

This notebook tests smarter decision rules, using the **out-of-fold probabilities of the tuning entities** to choose,
and the **validation** entities only to confirm:

1. Are the probabilities **calibrated** (does "0.8" really mean 80%)?
2. The precision / recall trade-off of the global threshold.
3. A **relative** rule: also require the pair to be close to the entity's best candidate.
4. An **expected-F0.5** rule: per entity, choose the number of IDs that maximises the expected score.
5. The **"one owner" rule**: each S2/S3 record belongs to at most one S1 entity.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.decision import expected_f05, global_threshold, relative_threshold, resolve_conflicts, to_submission
from src.evaluation import score_per_entity, score_report
from src.rule_experiment import load_query_sets

pd.set_option("display.width", 200)

predictions = pd.read_parquet(PROJECT_ROOT / "cache" / "phase9_predictions.parquet")
_, truth = load_query_sets()
tuning = predictions[predictions["query_set"] == "tuning"].reset_index(drop=True)
valid = predictions[predictions["query_set"] == "validation"].reset_index(drop=True)
print(f"tuning pairs {len(tuning):,} (out-of-fold probabilities) | validation pairs {len(valid):,}")

def f05(matches, truth_table):
    return score_report(to_submission(matches), truth_table)["macro_F0.5"]

tuning pairs 998,797 (out-of-fold probabilities) | validation pairs 999,057


---
## 1. Are the probabilities calibrated?

### Concept
A model is **calibrated** if, among all pairs it gives probability ≈ 0.8, about 80% are really matches.
Calibration doesn't matter for a simple threshold (only the ordering matters), but it **does** matter for rule 4,
which does arithmetic with the probabilities.

In [2]:
bins = pd.cut(tuning["probability"], [0, 0.02, 0.1, 0.3, 0.5, 0.7, 0.9, 0.98, 1.0], include_lowest=True)
calibration = tuning.groupby(bins, observed=True).agg(pairs=("label", "size"), mean_predicted=("probability", "mean"),
                                                      actual_share_true=("label", "mean"))
calibration.round(3)

,pairs,mean_predicted,actual_share_true
probability,,,
"(-0.001, 0.02]",915999,0.000,0.000
"(0.02, 0.1]",10711,0.046,0.070
"(0.1, 0.3]",5437,0.181,0.210
"(0.3, 0.5]",2829,0.395,0.416
"(0.5, 0.7]",2397,0.602,0.576
"(0.7, 0.9]",4036,0.817,0.791
"(0.9, 0.98]",6757,0.953,0.940
"(0.98, 1.0]",50631,0.998,0.997


Predicted and actual shares are close in every bucket → the probabilities are well calibrated
(gradient boosting with log-loss usually is). Note how few pairs are in the middle: the model is confident about most pairs.

---
## 2. The global threshold, in detail

For each threshold: pair precision, pair recall, and the macro F0.5 on the tuning entities (out-of-fold).

In [3]:
rows = []
for t in np.round(np.arange(0.40, 0.91, 0.025), 3):
    report = score_report(to_submission(global_threshold(tuning, t)), truth["tuning"])
    rows.append({"threshold": t, "macro F0.5": report["macro_F0.5"], "singletons F0.5": report["F0.5 on singletons"],
                 "pair precision": report["pair precision"], "pair recall": report["pair recall"]})
threshold_table = pd.DataFrame(rows).set_index("threshold").round(4)
best_global = threshold_table["macro F0.5"].idxmax()
print("best global threshold on tuning:", best_global)
threshold_table

best global threshold on tuning: 0.7


,macro F0.5,singletons F0.5,pair precision,pair recall
threshold,,,,
0.400,0.9229,0.8212,0.9518,0.8973
0.425,0.9246,0.8329,0.9549,0.8955
0.450,0.9258,0.8419,0.9575,0.8934
0.475,0.9271,0.8518,0.9599,0.8913
0.500,0.9281,0.8616,0.9621,0.8889
0.525,0.9287,0.8679,0.9640,0.8868
0.550,0.9293,0.8742,0.9659,0.8844
0.575,0.9301,0.8787,0.9680,0.8824
0.600,0.9306,0.8895,0.9700,0.8796


As the threshold rises, precision and the singleton score go up, recall goes down. F0.5 peaks where the last few
added matches start to cost more in precision than they give in recall.

---
## 3. Relative rule: also be close to the entity's best candidate

Idea: if an entity has a candidate at 0.99 and another at 0.72, the 0.72 one is less convincing than a 0.72
that is the entity's *best*. Rule: `probability >= t` **and** `probability >= ratio × best probability of the entity`.

In [4]:
rows = []
for t in [0.5, 0.6, 0.65, 0.7, 0.75]:
    for ratio in [0.0, 0.5, 0.7, 0.8, 0.9]:
        rows.append({"threshold": t, "ratio": ratio, "tuning F0.5": f05(relative_threshold(tuning, t, ratio), truth["tuning"])})
relative_table = pd.DataFrame(rows).pivot(index="threshold", columns="ratio", values="tuning F0.5").round(4)
relative_table

ratio,0.0,0.5,0.7,0.8,0.9
threshold,,,,,
0.50,0.9281,0.9281,0.9319,0.9313,0.9266
0.60,0.9306,0.9306,0.9322,0.9316,0.9269
0.65,0.9317,0.9317,0.9323,0.9318,0.9271
0.70,0.9324,0.9324,0.9324,0.9319,0.9273
0.75,0.9320,0.9320,0.9320,0.9315,0.9269


---
## 4. Expected-F0.5 rule (decision theory)

### Concept
For one entity with candidate probabilities `p1 ≥ p2 ≥ ...`, we can **estimate** the F0.5 we would get by predicting
its top 0, 1, 2, … candidates, and pick the best number:

- expected true matches `N ≈ p1 + p2 + ...` (divided by `1 − missing_rate`, since blocking misses ~6% of matches);
- expected correct among the top k: `TP ≈ p1 + ... + pk`;
- `F0.5(k) ≈ 1.25·TP / (k + 0.25·N)` for k ≥ 1, and `F0.5(0) = P(no match at all) ≈ (1−p1)(1−p2)…`.

### Why might it help?
A single global threshold ignores the **context** of each entity: the same 0.65 candidate can be worth predicting
for one entity and not for another. The example below shows how the rule reasons.

### Example

In [5]:
toy = pd.DataFrame({"source1_entity_id": ["A"] * 4 + ["B"] * 2 + ["C"],
                    "candidate_id": ["a1", "a2", "a3", "a4", "b1", "b2", "c1"],
                    "probability": [0.99, 0.98, 0.65, 0.05, 0.65, 0.10, 0.30]})
chosen = expected_f05(toy)
toy["predicted?"] = toy["candidate_id"].isin(chosen["candidate_id"])
toy

,source1_entity_id,candidate_id,probability,predicted?
0,A,a1,0.99,True
1,A,a2,0.98,True
2,A,a3,0.65,False
3,A,a4,0.05,False
4,B,b1,0.65,True
5,B,b2,0.10,False
6,C,c1,0.30,False


- **Entity A** (0.99, 0.98, 0.65, 0.05): the rule predicts only the top 2. Adding the 0.65 would raise expected recall a little
  but dilute an almost-perfect precision: expected F0.5 0.739 with 2 IDs vs 0.714 with 3.
- **Entity B** (0.65, 0.10): predicting nothing only pays if B has no match at all (≈ 0.35 × 0.90 = 0.32), while predicting b1
  gives an expected ≈ 0.55 → predict b1.
- **Entity C** (0.30): predicting nothing (expected 0.70) beats guessing c1.

In [6]:
rows = []
for missing_rate in [0.0, 0.03, 0.06, 0.10]:
    rows.append({"missing_rate": missing_rate, "tuning F0.5": f05(expected_f05(tuning, missing_rate), truth["tuning"])})
expected_table = pd.DataFrame(rows).set_index("missing_rate").round(4)
best_missing = expected_table["tuning F0.5"].idxmax()
expected_table

,tuning F0.5
missing_rate,
0.00,0.9317
0.03,0.9316
0.06,0.9316
0.10,0.9312


---
## 5. Comparison — chosen on tuning, confirmed on validation

In [7]:
best_rel_t, best_ratio = relative_table.stack().idxmax()
rules = {
    f"global threshold {best_global}": lambda d: global_threshold(d, best_global),
    f"relative (t={best_rel_t}, ratio={best_ratio})": lambda d: relative_threshold(d, best_rel_t, best_ratio),
    f"expected F0.5 (missing_rate={best_missing})": lambda d: expected_f05(d, best_missing),
}
rows = []
for name, rule in rules.items():
    t_report = score_report(to_submission(rule(tuning)), truth["tuning"])
    v_report = score_report(to_submission(rule(valid)), truth["validation"])
    rows.append({"rule": name, "tuning F0.5": t_report["macro_F0.5"], "validation F0.5": v_report["macro_F0.5"],
                 "validation singletons": v_report["F0.5 on singletons"], "validation matched": v_report["F0.5 on entities with matches"],
                 "pair precision": v_report["pair precision"], "pair recall": v_report["pair recall"]})
comparison = pd.DataFrame(rows).set_index("rule").round(4)
comparison

,tuning F0.5,validation F0.5,validation singletons,validation matched,pair precision,pair recall
rule,,,,,,
global threshold 0.7,0.9324,0.9340,0.9130,0.9353,0.9785,0.8714
"relative (t=0.7, ratio=0.0)",0.9324,0.9340,0.9130,0.9353,0.9785,0.8714
expected F0.5 (missing_rate=0.0),0.9317,0.9331,0.8357,0.9390,0.9779,0.8693


### Is the difference real? Paired bootstrap on the validation entities (as in Phase 6)

In [8]:
def per_entity(rule):
    return score_per_entity(to_submission(rule(valid)), truth["validation"]).set_index("source1_entity_id")["f05"]

base = per_entity(list(rules.values())[0])
rng = np.random.default_rng(0)
rows = []
for name, rule in list(rules.items())[1:]:
    diff = (per_entity(rule) - base).to_numpy()
    boots = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(1000)]
    rows.append({"rule vs global threshold": name, "mean difference": diff.mean(),
                 "95% CI low": np.percentile(boots, 2.5), "95% CI high": np.percentile(boots, 97.5),
                 "entities better": int((diff > 0).sum()), "entities worse": int((diff < 0).sum())})
pd.DataFrame(rows).set_index("rule vs global threshold").round(4)

,mean difference,95% CI low,95% CI high,entities better,entities worse
rule vs global threshold,,,,,
"relative (t=0.7, ratio=0.0)",0.000,0.0000,0.0000,0,0
expected F0.5 (missing_rate=0.0),-0.001,-0.0023,0.0002,274,429


**Result:** neither smarter rule beats the plain global threshold.
- The relative rule's best setting uses `ratio = 0` (i.e. it *is* the global threshold); a positive ratio never helps on tuning.
- The expected-F0.5 rule is slightly worse (−0.001, CI includes 0): it gains on entities with matches (0.939 vs 0.935) but loses on
  singletons (0.836 vs 0.913). Its "predict nothing" estimate `(1−p1)(1−p2)…` is too pessimistic — the plug-in approximation
  treats candidates as independent coin flips, while in reality a singleton's candidates are *all* look-alikes together.
- Lesson: a more sophisticated rule is only worth it if it **measurably** beats the simple one. Here it doesn't → keep the threshold.

---
## 6. The "one owner" rule

In the training ground truth **no** S2/S3 record is matched to more than one S1 entity (notebook 01).
So if two S1 entities both claim the same record, at most one can be right → keep only the highest-probability claim.

How often does that happen in our 40,000-entity experiment?

In [9]:
chosen_rule = comparison["tuning F0.5"].idxmax()
both = pd.concat([tuning, valid], ignore_index=True)
matches = rules[chosen_rule](both)
claims = matches.groupby("candidate_id").size()
print("predicted matches:", len(matches), "| records claimed by 2+ entities:", int((claims > 1).sum()))
resolved = resolve_conflicts(matches)
for name in ["tuning", "validation"]:
    before = f05(matches[matches["query_set"] == name], truth[name])
    after = f05(resolved[resolved["query_set"] == name], truth[name])
    print(f"{name}: F0.5 {before:.4f} -> {after:.4f} with conflict resolution")

predicted matches: 122860 | records claimed by 2+ entities: 18


tuning: F0.5 0.9324 -> 0.9325 with conflict resolution


validation: F0.5 0.9340 -> 0.9341 with conflict resolution


Our 40,000 query entities are only **1.8%** of all training S1 entities, so they rarely compete for the same record.
In the **test** set all 1.73M S1 entities are scored together, so look-alike entities (e.g. many "Family Center"s)
will compete for the same S2/S3 records much more often. The rule is logically safe (it can only remove a claim when a
*stronger* claim exists for the same record), and we will measure its real effect at full density in Phase 12
(full validation set = 441k entities, 20% of the training S1 density).

---
# Summary — Phase 10

| rule (chosen on tuning) | tuning F0.5 | validation F0.5 |
|---|---|---|
| **global threshold 0.70** | **0.9324** | **0.9340** |
| relative to the entity's best | 0.9324 (same rule) | 0.9340 |
| expected-F0.5 top-k | 0.9317 | 0.9331 |
| global 0.70 + one-owner rule | 0.9325 | 0.9341 |

**Decisions**
- Keep the **global threshold 0.70**. It is on a flat plateau (0.65–0.75 all within 0.0007 on tuning), so it is not fragile.
- Keep the **one-owner rule** (`src/decision.resolve_conflicts`): harmless here (18 conflicts among 40k entities),
  expected to matter more at full test density — to be measured in Phase 12.
- The probabilities are **well calibrated** (predicted ≈ actual in every bucket).

**Precision / recall at 0.70 (validation):** pair precision 0.979, pair recall 0.871 — F0.5's preference for precision in action.
Raising the threshold to 0.80 would push precision to 0.984 but lose more recall than it's worth.

**Next (Phase 11): error analysis** — look at the remaining false merges and missed matches, and categorise them.